# 🎬 Movie Genre Classification — Model Development

**CodSoft ML Internship — Task 1**

This notebook walks through the complete ML pipeline for predicting movie genres from title and plot description.

---

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import os
import time
import joblib
import warnings

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from scipy.sparse import hstack

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid')

print('All imports successful!')

## 2. Load Dataset

In [ ]:
# Load training data
train_df = pd.read_csv(
    '../data/train_data.txt', sep=' ::: ', engine='python',
    names=['ID', 'TITLE', 'GENRE', 'DESCRIPTION']
)

# Load test data (with solutions for evaluation)
test_df = pd.read_csv(
    '../data/test_data_solution.txt', sep=' ::: ', engine='python',
    names=['ID', 'TITLE', 'GENRE', 'DESCRIPTION']
)

print(f'Train samples: {len(train_df)}')
print(f'Test samples:  {len(test_df)}')
print(f'Train columns: {list(train_df.columns)}')

## 3. Dataset Inspection

In [ ]:
# Sample rows
train_df.head()

In [ ]:
# Basic info
print('Train DataFrame Info:')
print(train_df.info())
print()
print('Missing values:')
print(train_df.isnull().sum())

In [ ]:
# Genre distribution (original 27 genres)
print(f'Number of original genres: {train_df["GENRE"].nunique()}')
print()
print('Genre distribution (original):')
print(train_df['GENRE'].value_counts())

In [ ]:
# Visualize original genre distribution
fig, ax = plt.subplots(figsize=(12, 6))
train_df['GENRE'].value_counts().plot(kind='barh', color='steelblue', ax=ax)
ax.set_xlabel('Count')
ax.set_ylabel('Genre')
ax.set_title('Original Genre Distribution (27 classes)')
plt.tight_layout()
plt.show()

## 4. Genre Mapping & Preprocessing

In [ ]:
# Genre mapping: 27 original genres -> 10 clean classes
GENRE_MAPPING = {
    'action': 'Action/Adventure', 'adventure': 'Action/Adventure',
    'thriller': 'Action/Adventure', 'crime': 'Action/Adventure',
    'western': 'Action/Adventure', 'war': 'Action/Adventure',
    'comedy': 'Comedy',
    'drama': 'Drama', 'romance': 'Drama',
    'documentary': 'Documentary', 'biography': 'Documentary',
    'history': 'Documentary', 'news': 'Documentary',
    'horror': 'Horror/Mystery', 'mystery': 'Horror/Mystery',
    'sci-fi': 'Sci-Fi/Fantasy', 'fantasy': 'Sci-Fi/Fantasy',
    'family': 'Family/Animation', 'animation': 'Family/Animation',
    'game-show': 'Entertainment', 'reality-tv': 'Entertainment',
    'talk-show': 'Entertainment', 'music': 'Entertainment',
    'musical': 'Entertainment', 'sport': 'Entertainment',
    'short': 'Short Film',
    'adult': 'Adult',
}

# Apply mapping
train_df['GENRE'] = train_df['GENRE'].str.strip().str.lower().map(GENRE_MAPPING)
test_df['GENRE'] = test_df['GENRE'].str.strip().str.lower().map(GENRE_MAPPING)

# Drop unmapped
train_df = train_df.dropna(subset=['GENRE']).reset_index(drop=True)
test_df = test_df.dropna(subset=['GENRE']).reset_index(drop=True)

print(f'Mapped to {train_df["GENRE"].nunique()} classes:')
print(train_df['GENRE'].value_counts())

In [ ]:
# Visualize mapped genre distribution
fig, ax = plt.subplots(figsize=(10, 5))
colors = sns.color_palette('viridis', n_colors=train_df['GENRE'].nunique())
train_df['GENRE'].value_counts().plot(kind='barh', color=colors, ax=ax)
ax.set_xlabel('Count')
ax.set_ylabel('Genre')
ax.set_title('Mapped Genre Distribution (10 classes)')
plt.tight_layout()
plt.show()

## 5. Text Cleaning

In [ ]:
def clean_text(text):
    """Clean and normalize text."""
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Combine Title + Description and clean
train_df['TEXT'] = (train_df['TITLE'] + ' ' + train_df['DESCRIPTION']).apply(clean_text)
test_df['TEXT'] = (test_df['TITLE'] + ' ' + test_df['DESCRIPTION']).apply(clean_text)

# Show sample
print('Sample cleaned text:')
print(train_df['TEXT'].iloc[0][:200])

In [ ]:
# Text length distribution
train_df['TEXT_LEN'] = train_df['TEXT'].apply(len)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(train_df['TEXT_LEN'], bins=50, color='mediumpurple', edgecolor='white', alpha=0.8)
ax.set_xlabel('Character Count')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Text Lengths')
ax.axvline(train_df['TEXT_LEN'].mean(), color='red', linestyle='--', label=f'Mean: {train_df["TEXT_LEN"].mean():.0f}')
ax.legend()
plt.tight_layout()
plt.show()

## 6. TF-IDF Feature Extraction

In [ ]:
# Word-level TF-IDF
word_vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=80000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=3,
    max_df=0.7,
    dtype=np.float32,
)

# Character-level TF-IDF
char_vectorizer = TfidfVectorizer(
    analyzer='char_wb',
    max_features=50000,
    ngram_range=(3, 5),
    sublinear_tf=True,
    min_df=3,
    max_df=0.7,
    dtype=np.float32,
)

print('Fitting word-level TF-IDF...')
X_train_word = word_vectorizer.fit_transform(train_df['TEXT'])
X_test_word = word_vectorizer.transform(test_df['TEXT'])
print(f'  Word features: {X_train_word.shape[1]}')

print('Fitting char-level TF-IDF...')
X_train_char = char_vectorizer.fit_transform(train_df['TEXT'])
X_test_char = char_vectorizer.transform(test_df['TEXT'])
print(f'  Char features: {X_train_char.shape[1]}')

# Combine
X_train = hstack([X_train_word, X_train_char])
X_test = hstack([X_test_word, X_test_char])

y_train = train_df['GENRE']
y_test = test_df['GENRE']

print(f'\nTotal features: {X_train.shape[1]}')
print(f'Train shape: {X_train.shape}')
print(f'Test shape:  {X_test.shape}')

## 7. Model Training & Comparison

In [ ]:
# Define models
models = {
    'Logistic Regression': LogisticRegression(C=1.0, max_iter=1000, n_jobs=-1, solver='lbfgs'),
    'LinearSVC': LinearSVC(C=1.0, max_iter=2000),
    'MultinomialNB': MultinomialNB(alpha=0.1),
}

results = {}

for name, model in models.items():
    print(f'\nTraining {name}...')
    t = time.time()
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    elapsed = time.time() - t
    results[name] = {'accuracy': acc, 'time': elapsed, 'model': model, 'predictions': preds}
    print(f'  Accuracy: {acc:.4f}  |  Time: {elapsed:.2f}s')

In [ ]:
# Comparison bar chart
model_names = list(results.keys())
accuracies = [results[n]['accuracy'] for n in model_names]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(model_names, accuracies, color=['#6d28d9', '#3b82f6', '#10b981'])
ax.set_xlabel('Accuracy')
ax.set_title('Model Comparison — Test Accuracy')
ax.set_xlim(0.5, max(accuracies) + 0.05)

for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{acc:.4f}', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 8. Best Model Evaluation

In [ ]:
# Select best model
best_name = max(results, key=lambda n: results[n]['accuracy'])
best_model = results[best_name]['model']
best_preds = results[best_name]['predictions']
best_acc = results[best_name]['accuracy']

print(f'Best Model: {best_name}')
print(f'Test Accuracy: {best_acc:.4f}')
print()
print('Classification Report:')
print(classification_report(y_test, best_preds))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, best_preds, labels=sorted(y_test.unique()))

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=sorted(y_test.unique()),
            yticklabels=sorted(y_test.unique()), ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title(f'Confusion Matrix — {best_name}')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 9. Sample Predictions

In [ ]:
# Test with sample movies
samples = [
    ('The Matrix', 'A computer hacker learns from mysterious rebels about the true nature of his reality and his role in the war against its controllers.'),
    ('The Conjuring', 'Paranormal investigators work to help a family terrorized by a dark presence in their farmhouse.'),
    ('Toy Story', 'A cowboy doll is profoundly threatened and jealous when a new spaceman figure supplants him as top toy in a boys room.'),
    ('The Wolf of Wall Street', 'Based on the true story of Jordan Belfort, from his rise to a wealthy stock-broker living the high life to his fall.'),
    ('Titanic', 'A seventeen-year-old aristocrat falls in love with a kind but poor artist aboard the luxurious, ill-fated R.M.S. Titanic.'),
]

print('Sample Predictions:')
print('-' * 60)
for title, desc in samples:
    text = clean_text(title + ' ' + desc)
    w_feat = word_vectorizer.transform([text])
    c_feat = char_vectorizer.transform([text])
    feat = hstack([w_feat, c_feat])
    pred = best_model.predict(feat)[0]
    print(f'  {title:30s} -> {pred}')

## 10. Save Final Model

In [ ]:
# Save model bundle (model + both vectorizers in a single file)
os.makedirs('../models', exist_ok=True)

model_bundle = {
    'model': best_model,
    'word_vectorizer': word_vectorizer,
    'char_vectorizer': char_vectorizer,
    'model_name': best_name,
    'accuracy': best_acc,
    'all_results': {n: results[n]['accuracy'] for n in results},
}

model_path = '../models/movie_genre_model.pkl'
joblib.dump(model_bundle, model_path)

print(f'Model bundle saved to: {model_path}')
print(f'Best model: {best_name} (Accuracy: {best_acc:.4f})')
print(f'Bundle contains: model, word_vectorizer, char_vectorizer')

---

## Summary

- **Dataset**: 54,214 train / 54,200 test movie descriptions
- **Preprocessing**: 27 genres mapped to 10 clean classes, text cleaned (lowercase, special chars removed)
- **Features**: Dual TF-IDF — 80K word-level (1,2)-grams + 50K char-level (3,5)-grams
- **Models compared**: Logistic Regression, LinearSVC, MultinomialNB
- **Best model selected automatically** based on test accuracy
- **Final model saved** as a single `.pkl` bundle for the Streamlit app